In [2]:
pip install chromadb

  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.45.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached orjson-3.13.0-cp311-cp311-win_amd64.whl.metadata (43 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached aiohttp-3.14.5-cp311-cp311-win_amd64.whl.metadata (8.5 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached multidict-7.2.1-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached propcache-0.5.4-cp

In [3]:
import chromadb

In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

d:\itw\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]


In [5]:
client = chromadb.PersistentClient(
    path="D:\\itw\\chroma"
)

In [6]:
collection = client.get_or_create_collection(
    name="itw8",
    metadata={"hnsw:space": "cosine"},
)

In [13]:
documents = [
    "keshav is the best ai engineer",
    "he is from bihar",
    "he is a good person",
    "he is a good friend",
    "he is a good human being",
    "he is talented and studies in presidency university",
    "he is a gambling addict "
]

document_ids = [f"doc_{i}" for i in range(len(documents))]

metadata = [{"source": f"source_{i}"} for i in range(len(documents))]



In [14]:
documents_embeddings = embedding_model.encode(documents)

print("Document embeddings shape:", documents_embeddings.shape)

Document embeddings shape: (7, 384)


In [16]:
# Store the documents and their embeddings in ChromaDB
collection.upsert(
    ids=document_ids,
    documents=documents,
    embeddings=documents_embeddings.tolist(),
    metadatas=metadata,
)

print(f"Stored {len(document_ids)} documents in '{collection.name}'.")

Stored 7 documents in 'itw8'.


In [21]:
def search_documents(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    return results

In [22]:
while True:
    user_query = input("Enter your query (or 'exit' to quit): ")
    if user_query.lower() == 'exit':
        break

    search_results = search_documents(user_query, top_k=3)

    print("\nSearch Results:")
    for i, (doc, metadata, distance) in enumerate(zip(search_results['documents'][0], search_results['metadatas'][0], search_results['distances'][0])):
        print(f"Result {i + 1}:")
        print(f"Document: {doc}")
        print(f"Metadata: {metadata}")
        print(f"Distance: {distance}\n")


Search Results:
Result 1:
Document: he is a good person
Metadata: {'source': 'source_2'}
Distance: 0.7653684616088867

Result 2:
Document: he is a good friend
Metadata: {'source': 'source_3'}
Distance: 0.7829506993293762

Result 3:
Document: he is a good human being
Metadata: {'source': 'source_4'}
Distance: 0.7935287952423096


Search Results:
Result 1:
Document: he is a gambling addict 
Metadata: {'source': 'source_6'}
Distance: 0.4954342246055603

Result 2:
Document: keshav is the best ai engineer
Metadata: {'source': 'source_0'}
Distance: 0.5647110939025879

Result 3:
Document: he is a good person
Metadata: {'source': 'source_2'}
Distance: 0.753836989402771

